# Phase 9 — Natural Language to SQL

This notebook implements the Natural Language to SQL component
of the Databricks GenAI Data Analyst Copilot.

Flow:

User Question
    ↓
Question Classification
    ↓
Schema Retrieval
    ↓
Prompt Construction
    ↓
LLM
    ↓
Structured SQL Plan
    ↓
Phase 10 SQL Validation

The LLM is never allowed to execute SQL directly.

In [0]:
import json
import re
from datetime import datetime

from pyspark.sql import functions as F

In [0]:
CATALOG = "genai_copilot"
SCHEMA_METADATA_TABLE = "genai_copilot.gold.schema_metadata"

print("Catalog:", CATALOG)
print("Schema metadata:", SCHEMA_METADATA_TABLE)

In [0]:
schema_metadata_df = spark.table(
    SCHEMA_METADATA_TABLE
)

print(
    "Schema metadata rows:",
    schema_metadata_df.count()
)

display(schema_metadata_df.limit(20))

In [0]:
schema_rows = (
    schema_metadata_df
    .select(
        "table_name",
        "column_name",
        "data_type",
        "description"
    )
    .orderBy(
        "table_name",
        "column_name"
    )
    .collect()
)

schema_catalog = {}

for row in schema_rows:

    table_name = row["table_name"]

    if table_name not in schema_catalog:
        schema_catalog[table_name] = []

    schema_catalog[table_name].append({
        "column_name": row["column_name"],
        "data_type": row["data_type"],
        "description": row["description"]
    })

print("Tables available to GenAI:")

for table_name in schema_catalog:
    print("-", table_name)

In [0]:
ALLOWED_TABLES = [
    "genai_copilot.silver.sales",
    "genai_copilot.gold.monthly_sales",
    "genai_copilot.gold.region_sales",
    "genai_copilot.gold.customer_metrics",
    "genai_copilot.gold.product_metrics",
    "genai_copilot.gold.category_metrics",
]

available_allowed_tables = [
    table_name
    for table_name in ALLOWED_TABLES
    if table_name in schema_catalog
]

print("Allowed analytical tables:")

for table_name in available_allowed_tables:
    print("-", table_name)

In [0]:
def format_schema_for_llm(table_names):
    
    sections = []

    for table_name in table_names:

        if table_name not in schema_catalog:
            continue

        lines = [
            f"TABLE: {table_name}"
        ]

        for column in schema_catalog[table_name]:

            description = (
                column["description"]
                or "No description available"
            )

            lines.append(
                f"- {column['column_name']} "
                f"({column['data_type']}): "
                f"{description}"
            )

        sections.append(
            "\n".join(lines)
        )

    return "\n\n".join(sections)

In [0]:
schema_text = format_schema_for_llm(
    available_allowed_tables
)

print(schema_text)

In [0]:
INTENTS = [
    "aggregation",
    "comparison",
    "trend",
    "ranking",
    "filtering",
    "anomaly",
    "descriptive_statistics",
    "visualization",
    "unsupported"
]

print(INTENTS)

In [0]:
def classify_question(question):

    q = question.lower()

    if any(
        word in q
        for word in [
            "top",
            "highest",
            "lowest",
            "best",
            "worst",
            "ranking"
        ]
    ):
        return "ranking"

    if any(
        word in q
        for word in [
            "month",
            "monthly",
            "over time",
            "trend",
            "change over time"
        ]
    ):
        return "trend"

    if any(
        word in q
        for word in [
            "compare",
            "difference",
            "versus",
            "vs"
        ]
    ):
        return "comparison"

    if any(
        word in q
        for word in [
            "average",
            "mean",
            "median",
            "minimum",
            "maximum",
            "standard deviation"
        ]
    ):
        return "descriptive_statistics"

    if any(
        word in q
        for word in [
            "show",
            "display",
            "visualize",
            "chart",
            "graph"
        ]
    ):
        return "visualization"

    if any(
        word in q
        for word in [
            "why",
            "decline",
            "decrease",
            "drop",
            "increase",
            "anomaly"
        ]
    ):
        return "anomaly"

    if any(
        word in q
        for word in [
            "how many",
            "how much",
            "total",
            "sum"
        ]
    ):
        return "aggregation"

    return "filtering"

In [0]:
test_questions = [
    "What is total revenue?",
    "Which region generated the highest revenue?",
    "What is monthly revenue?",
    "What are the top 10 customers by revenue?",
    "What is the average order value?",
    "Show revenue by region",
]

for question in test_questions:

    intent = classify_question(question)

    print(
        f"Question: {question}"
    )

    print(
        f"Intent: {intent}"
    )

    print("-" * 60)

In [0]:
def select_candidate_tables(question):

    q = question.lower()

    candidates = []

    if any(
        word in q
        for word in [
            "monthly",
            "month",
            "trend",
            "over time"
        ]
    ):
        if "genai_copilot.gold.monthly_sales" in available_allowed_tables:
            candidates.append(
                "genai_copilot.gold.monthly_sales"
            )

    if "region" in q:
        if "genai_copilot.gold.region_sales" in available_allowed_tables:
            candidates.append(
                "genai_copilot.gold.region_sales"
            )

    if "customer" in q:
        if "genai_copilot.gold.customer_metrics" in available_allowed_tables:
            candidates.append(
                "genai_copilot.gold.customer_metrics"
            )

    if "product" in q:
        if "genai_copilot.gold.product_metrics" in available_allowed_tables:
            candidates.append(
                "genai_copilot.gold.product_metrics"
            )

    if "category" in q:
        if "genai_copilot.gold.category_metrics" in available_allowed_tables:
            candidates.append(
                "genai_copilot.gold.category_metrics"
            )

    if not candidates:

        if "genai_copilot.silver.sales" in available_allowed_tables:
            candidates.append(
                "genai_copilot.silver.sales"
            )

    return candidates

In [0]:
for question in test_questions:

    candidates = select_candidate_tables(
        question
    )

    print("Question:", question)
    print("Candidate tables:")

    for table in candidates:
        print("  -", table)

    print("-" * 60)

In [0]:
def build_sql_prompt(
    question,
    intent,
    candidate_tables
):

    schema_text = format_schema_for_llm(
        candidate_tables
    )

    prompt = f"""
You are a senior data analyst working with Databricks.

Your task is to convert a natural-language business question
into a safe analytical SQL query.

USER QUESTION:
{question}

INTENT:
{intent}

AVAILABLE SCHEMA:

{schema_text}

RULES:

1. Generate read-only SQL.
2. Only use tables listed in the schema.
3. Only use columns listed in the schema.
4. Do not invent columns.
5. Do not invent tables.
6. Do not modify data.
7. Do not use INSERT.
8. Do not use UPDATE.
9. Do not use DELETE.
10. Do not use DROP.
11. Do not use ALTER.
12. Do not use CREATE.
13. Do not use TRUNCATE.
14. Do not use GRANT.
15. Do not use REVOKE.
16. Return one SQL statement only.
17. Prefer aggregation tables when appropriate.
18. Use aliases for calculated metrics.
19. Add a LIMIT when returning detailed records.
20. Do not fabricate numerical results.

Return ONLY valid JSON.

Expected format:

{{
    "intent": "...",
    "table": "...",
    "columns": ["..."],
    "sql": "...",
    "chart": "...",
    "explanation": "..."
}}
"""

    return prompt

In [0]:
question = "Which region generated the highest revenue?"

intent = classify_question(
    question
)

candidate_tables = select_candidate_tables(
    question
)

prompt = build_sql_prompt(
    question,
    intent,
    candidate_tables
)

print(prompt)

In [0]:
def call_llm(prompt):
    """
    Central LLM interface.

    This function intentionally isolates the model dependency
    from the rest of the SQL generation pipeline.

    The actual Databricks model endpoint will be connected
    here once an available model endpoint is confirmed.
    """

    raise NotImplementedError(
        "LLM endpoint is not configured yet. "
        "Use the deterministic test mode until the "
        "Databricks model endpoint is configured."
    )

In [0]:
MOCK_LLM = True

In [0]:
def mock_llm_response(question):

    q = question.lower()

    if "highest revenue" in q and "region" in q:

        return {
            "intent": "ranking",
            "table": "genai_copilot.gold.region_sales",
            "columns": [
                "region",
                "total_revenue"
            ],
            "sql": """
SELECT
    region,
    total_revenue
FROM genai_copilot.gold.region_sales
ORDER BY total_revenue DESC
LIMIT 1
""".strip(),
            "chart": "bar",
            "explanation": (
                "Ranks regions by total revenue "
                "and returns the highest-revenue region."
            )
        }

    if "total revenue" in q:

        return {
            "intent": "aggregation",
            "table": "genai_copilot.silver.sales",
            "columns": [
                "revenue"
            ],
            "sql": """
SELECT
    SUM(revenue) AS total_revenue
FROM genai_copilot.silver.sales
""".strip(),
            "chart": "number",
            "explanation": (
                "Calculates total revenue across all sales."
            )
        }

    if "monthly revenue" in q:

        return {
            "intent": "trend",
            "table": "genai_copilot.gold.monthly_sales",
            "columns": [
                "month",
                "total_revenue"
            ],
            "sql": """
SELECT
    month,
    total_revenue
FROM genai_copilot.gold.monthly_sales
ORDER BY month
""".strip(),
            "chart": "line",
            "explanation": (
                "Returns revenue by month to show "
                "the revenue trend over time."
            )
        }

    return {
        "intent": "unsupported",
        "table": None,
        "columns": [],
        "sql": None,
        "chart": "table",
        "explanation": (
            "The requested question is not supported "
            "by the current test generator."
        )
    }

In [0]:
def generate_sql_plan(question):

    intent = classify_question(
        question
    )

    candidate_tables = select_candidate_tables(
        question
    )

    prompt = build_sql_prompt(
        question,
        intent,
        candidate_tables
    )

    if MOCK_LLM:

        response = mock_llm_response(
            question
        )

    else:

        response = call_llm(
            prompt
        )

    return {
        "question": question,
        "intent": intent,
        "candidate_tables": candidate_tables,
        "prompt": prompt,
        "response": response,
        "generated_at": datetime.now()
    }

In [0]:
question = "Which region generated the highest revenue?"

result = generate_sql_plan(
    question
)

print(
    json.dumps(
        result["response"],
        indent=2
    )
)

In [0]:
def validate_sql_plan_structure(plan):

    required_fields = [
        "intent",
        "table",
        "columns",
        "sql",
        "chart",
        "explanation"
    ]

    missing_fields = [
        field
        for field in required_fields
        if field not in plan
    ]

    if missing_fields:
        return False, (
            f"Missing fields: {missing_fields}"
        )

    return True, "Valid structure"

In [0]:
is_valid, message = validate_sql_plan_structure(
    result["response"]
)

print("Valid:", is_valid)
print("Message:", message)

In [0]:
demo_questions = [
    "What is total revenue?",
    "Which region generated the highest revenue?",
    "What is monthly revenue?",
]

for question in demo_questions:

    result = generate_sql_plan(
        question
    )

    plan = result["response"]

    valid, message = (
        validate_sql_plan_structure(plan)
    )

    print("=" * 70)
    print("QUESTION:", question)
    print("INTENT:", plan["intent"])
    print("TABLE:", plan["table"])
    print("VALID:", valid)
    print("SQL:")
    print(plan["sql"])

In [0]:
generation_log = []

for question in demo_questions:

    result = generate_sql_plan(
        question
    )

    plan = result["response"]

    generation_log.append({
        "question": question,
        "intent": plan["intent"],
        "table": plan["table"],
        "sql": plan["sql"],
        "chart": plan["chart"],
        "explanation": plan["explanation"],
        "generated_at": result["generated_at"]
    })

print(
    f"Generated {len(generation_log)} SQL plans."
)

In [0]:
successful = 0

for item in generation_log:

    if item["sql"]:
        successful += 1

total = len(generation_log)

print("=" * 60)
print("PHASE 9 VALIDATION")
print("=" * 60)
print("Questions tested:", total)
print("SQL plans generated:", successful)

if successful == total:
    print("STATUS: PASS")
else:
    print("STATUS: FAIL")